<a href="https://www.kaggle.com/code/dozeradi007/dual-sr-project-patch-creation?scriptVersionId=338339952" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import rasterio
import os
from rasterio.windows import Window

In [2]:
def create_patch_dataset(input_dataset, output_dataset):
    patch_sizes = {
        "HR_0.5m": 512,
        "LR_1m": 256,
        "LR_2m": 128
    }

    splits = ["train", "val", "test"]

    for split in splits:
        print(f"\nProcessing {split}...")

        for resolution, patch_size in patch_sizes.items():

            input_dir = os.path.join(input_dataset, split, resolution)

            if not os.path.exists(input_dir):
                print(f"Skipping missing folder: {input_dir}")
                continue

            image_list = sorted(os.listdir(input_dir))

            # creating image path
            for image_name in image_list:

                image_path = os.path.join(input_dir, image_name)

                if not os.path.isfile(image_path):
                    continue

                image_id = os.path.splitext(image_name)[0]

                output_dir = os.path.join(
                    output_dataset,
                    split,
                    resolution,
                    image_id
                )
                os.makedirs(output_dir, exist_ok=True)

                # copying image
                with rasterio.open(image_path) as src:

                    width = src.width
                    height = src.height

                    patch_id = 1

                    for y in range(0, height, patch_size):
                        for x in range(0, width, patch_size):

                            window = Window(
                                col_off=x,
                                row_off=y,
                                width=min(patch_size, width - x),
                                height=min(patch_size, height - y)
                            )

                            patch = src.read(window=window)

                            profile = src.profile.copy()
                            profile.update(
                                width=patch.shape[2],
                                height=patch.shape[1],
                                transform=src.window_transform(window)
                            )

                            patch_name = f"patch_{patch_id:03d}.tif"

                            with rasterio.open(
                                os.path.join(output_dir, patch_name),
                                "w",
                                **profile
                            ) as dst:
                                dst.write(patch)

                            patch_id += 1

                # print(f"{split}/{resolution}/{image_id} -> {patch_id-1} patches")
            print(f"{split}/{resolution} : patches created")

    print("\nPatch dataset created successfully!")

In [3]:
input_dataset = "/kaggle/input/datasets/dozeradi007/paired-hr-lr-satellite-imagery-dataset/dataset"
output_dataset = "/kaggle/working/patch_dataset"

create_patch_dataset(input_dataset, output_dataset)


Processing train...
train/HR_0.5m : patches created
train/LR_1m : patches created
train/LR_2m : patches created

Processing val...
val/HR_0.5m : patches created
val/LR_1m : patches created
val/LR_2m : patches created

Processing test...
test/HR_0.5m : patches created
test/LR_1m : patches created
test/LR_2m : patches created

Patch dataset created successfully!
